In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

<details>
<summary><b>Paketversionen</b></summary>

Der Code auf dieser Seite wurde mit den folgenden Anforderungen entwickelt.
Wir empfehlen, diese Versionen oder neuere zu verwenden.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>
Neben dem [Visualisieren von Anweisungen in einem Circuit](/guides/visualize-circuits) möchtest du vielleicht auch das Scheduling eines Circuits visualisieren, indem du die Qiskit-Methode [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer) verwendest. Diese Visualisierung kann dir dabei helfen, beispielsweise Leerlaufzeiten auf Qubits schnell zu erkennen. Allerdings liefert diese Methode für dynamische Circuits keine genauen Ergebnisse. Um das Scheduling dynamischer Circuits zu visualisieren, verwende die Methode `draw_circuit_schedule_timing`, wie im Abschnitt [Qiskit Runtime-Unterstützung](#qr-support) beschrieben.

## Beispiele

Um ein geplantes Circuit-Programm zu visualisieren, kannst du diese Funktion mit einer Reihe von Steuerargumenten aufrufen. Der Großteil des Erscheinungsbilds des Ausgabebilds lässt sich über ein Stylesheet anpassen, was jedoch nicht zwingend erforderlich ist.

### Mit dem Standard-Stylesheet zeichnen

In [1]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)

backend = GenericBackendV2(5)

pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)

draw(isa_circuit, target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/visualize-circuit-timing/extracted-outputs/5b21b8c1-0.svg)

### Mit einem für das Programm-Debugging geeigneten Stylesheet zeichnen

In [2]:
from qiskit import QuantumCircuit
from qiskit.visualization.timeline import draw, IQXDebugging
from qiskit.providers.fake_provider import GenericBackendV2
from qiskit.transpiler import generate_preset_pass_manager

qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

backend = GenericBackendV2(5)
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
isa_circuit = pm.run(qc)
draw(isa_circuit, style=IQXDebugging(), target=backend.target)

<Image src="../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg" alt="Output of the previous code cell" />

![Ausgabe der vorherigen Code-Zelle](../docs/images/guides/visualize-circuit-timing/extracted-outputs/907dc46c-0.svg)

Du kannst benutzerdefinierte Generator- oder Layout-Funktionen erstellen und ein vorhandenes Stylesheet mit diesen Funktionen aktualisieren. Auf diese Weise kannst du den Großteil des Erscheinungsbilds des Ausgabebilds steuern, ohne die Codebasis des geplanten Circuit-Zeichners zu ändern. Weitere Beispiele findest du in der API-Referenz für [`timeline_drawer`](https://docs.quantum.ibm.com/api/qiskit/qiskit.visualization.timeline_drawer).
<span id="qr-support"></span>
## Qiskit Runtime-Unterstützung
Während der in Qiskit integrierte Timeline-Drawer für statische Circuits nützlich ist, spiegelt er das Timing von [dynamischen Circuits](/guides/classical-feedforward-and-control-flow) möglicherweise nicht korrekt wider, da implizite Operationen wie Broadcasting und Branch-Bestimmung auftreten. Als Teil der Unterstützung für dynamische Circuits gibt Qiskit Runtime auf Anfrage die genauen Circuit-Timing-Informationen in den Job-Ergebnissen zurück.

> **Note:** - Dies ist eine experimentelle Funktion. Sie befindet sich im Vorschau-Status und kann daher geändert werden.
> - Diese Funktion gilt nur für Qiskit Runtime Sampler-Jobs.
> - Obwohl die gesamte Circuit-Zeit in den „compilation"-Metadaten zurückgegeben wird, ist dies NICHT die für die Abrechnung verwendete Zeit (Quantenzeit).
### Timing-Datenabruf aktivieren
Um den Timing-Datenabruf zu aktivieren, setze das experimentelle Flag `scheduler_timing` beim Ausführen des Primitive-Jobs auf `True`.